In [1]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn

# Ensure path to src/
sys.path.append(os.path.abspath('..'))
from src.lstm_encoder import HybridCNNBiLSTMEncoder
from src.ssl_joint import DualPretextSSLModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing Phase 11 Backbone Extraction on: {device}")

# 1. Paths Setup
saved_models_dir = Path('../saved_models')
saved_models_dir.mkdir(parents=True, exist_ok=True)

source_joint_checkpoint = saved_models_dir / 'ssl_joint_dual_pretext_full.pth'
target_clean_checkpoint = saved_models_dir / 'ssl_cnn_bilstm.pth'

assert source_joint_checkpoint.exists(), f"Source checkpoint not found at: {source_joint_checkpoint}"

# 2. Instantiate Full Phase 10 Model Architecture and Load Weights
joint_model = DualPretextSSLModel(
    in_channels=25,
    cnn_hidden_dims=(64, 128),
    lstm_hidden_dim=128,
    lstm_layers=2,
    proj_dim=64,
    mask_ratio=0.20
).to(device)

print(f"Loading full Phase 10 multi-task checkpoint from: {source_joint_checkpoint}")
joint_model.load_state_dict(torch.load(source_joint_checkpoint, map_location=device))
joint_model.eval()

# 3. Extract Only the Encoder Backbone
encoder_backbone = joint_model.encoder
encoder_state_dict = encoder_backbone.state_dict()

# 4. Save to the Standardized Phase 11 Path
torch.save(encoder_state_dict, target_clean_checkpoint)
print(f"\n[SUCCESS] Clean Pretrained Encoder successfully saved to: {target_clean_checkpoint}")

# -------------------------------------------------------------
# 5. Sanity Check & Verification Suite
# Load into a clean, standalone HybridCNNBiLSTMEncoder instance
# -------------------------------------------------------------
print("\n--- Verifying Saved Pretrained Encoder Checkpoint ---")

clean_encoder = HybridCNNBiLSTMEncoder(
    in_channels=25,
    cnn_hidden_dims=(64, 128),
    lstm_hidden_dim=128,
    lstm_layers=2,
    pool_strategy="mean"
).to(device)

# Load state dict strictly into the clean encoder
clean_encoder.load_state_dict(torch.load(target_clean_checkpoint, map_location=device))
clean_encoder.eval()

# Compare parameter checksums between source model encoder and clean standalone encoder
mismatches = 0
for (name_src, param_src), (name_clean, param_clean) in zip(
    encoder_backbone.named_parameters(), clean_encoder.named_parameters()
):
    if not torch.allclose(param_src, param_clean):
        mismatches += 1
        print(f"Mismatch detected in parameter: {name_clean}")

assert mismatches == 0, f"Found {mismatches} parameter discrepancies!"
print("Weight Identity Verification: PASS (100% parameter checksum match)")

# Forward pass validation on synthetic batch
dummy_input = torch.randn(8, 48, 25).to(device)
with torch.no_grad():
    latent_out = clean_encoder(dummy_input)

print(f"Input Verification Shape:  {tuple(dummy_input.shape)}")
print(f"Output Latent Space Shape: {tuple(latent_out.shape)} (Expected: (8, 256))")
assert latent_out.shape == (8, 256), "Dimension mismatch in standalone forward pass!"

# Inspect model parameter footprint
total_params = sum(p.numel() for p in clean_encoder.parameters())
trainable_params = sum(p.numel() for p in clean_encoder.parameters() if p.requires_grad)
print(f"Isolated Encoder Total Parameters:     {total_params:,}")
print(f"Isolated Encoder Trainable Parameters: {trainable_params:,}")
print(f"Checkpoint File Size:                  {target_clean_checkpoint.stat().st_size / 1e6:.2f} MB")

Executing Phase 11 Backbone Extraction on: cuda
Loading full Phase 10 multi-task checkpoint from: ..\saved_models\ssl_joint_dual_pretext_full.pth

[SUCCESS] Clean Pretrained Encoder successfully saved to: ..\saved_models\ssl_cnn_bilstm.pth

--- Verifying Saved Pretrained Encoder Checkpoint ---
Weight Identity Verification: PASS (100% parameter checksum match)


C:\Users\waliu\AppData\Local\Temp\ipykernel_20836\1783374177.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  joint_model.load_state_dict(torch.load(source_joint_checkpo

Input Verification Shape:  (8, 48, 25)
Output Latent Space Shape: (8, 256) (Expected: (8, 256))
Isolated Encoder Total Parameters:     699,392
Isolated Encoder Trainable Parameters: 699,392
Checkpoint File Size:                  2.81 MB
